# ProductIQ — AI Operations
## LangChain · RAG · CrewAI

All AI logic for the ProductIQ retail decision system in one notebook.
Egyptian market context — bilingual Arabic/English.

**Stack:** Groq (primary LLM) · NVIDIA & Google (alternatives) · FAISS + HuggingFace embeddings · CrewAI multi-agent board meeting

In [ ]:
# Install packages (run once)
!pip install -q langchain langchain-core langchain-groq langchain-community langchain-huggingface
!pip install -q faiss-cpu sentence-transformers crewai pandas python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# ─── API KEYS ───
# Keys are loaded from ../backend/.env (git-ignored). NEVER hardcode keys here.
load_dotenv('../backend/.env')

# Fallback: paste your own keys below ONLY for local runs, and clear before committing.
if not os.getenv('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = 'PASTE-YOUR-KEY-HERE'

print('API keys loaded ✓' if os.getenv('GROQ_API_KEY') else 'No keys found — check ../backend/.env')

In [ ]:
# ─── LLM setup (Groq primary) ───
from langchain_groq import ChatGroq

llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.2)

# Quick smoke test
print(llm.invoke('Say "ProductIQ ready" in Arabic and English.').content)

---
## Part 1: LangChain — Structured Recommendations
Groq analyzes retail data and returns structured JSON recommendations (Arabic or English).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List

# ─── Structured output schema ───
class Recommendation(BaseModel):
    product: str = Field(description='Product name')
    action: str = Field(description='Action: restock / discount / bundle / remove')
    reason: str = Field(description='Why this action is recommended')
    confidence: int = Field(description='Confidence 0-100')

class RetailAnalysis(BaseModel):
    summary: str = Field(description='One-paragraph executive summary')
    recommendations: List[Recommendation]

parser = JsonOutputParser(pydantic_object=RetailAnalysis)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are an AI retail analyst for the Egyptian market. '
     'Analyze the store data and return structured recommendations. '
     'Use EGP currency. Respond in {language}.\n'
     '{format_instructions}'),
    ('human', 'Sales data:\n{sales_data}\n\nInventory data:\n{inventory_data}')
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser
print('Chain ready')

In [ ]:
# ─── Sample Egyptian store data ───
sample_sales = '''Product,Qty Sold,Revenue EGP,Cost EGP
Samsung Galaxy A56,45,269550,229500
iPhone 16 Pro,12,479880,384000
Xiaomi Redmi Note 14,78,140400,117000
Sony WH-1000XM6,8,119920,84000
Samsung Galaxy Tab S10,15,224850,172500'''

sample_inventory = '''Product,Stock,Cost EGP,Price EGP,Supplier
Samsung Galaxy A56,120,5100,5990,النور للتوريدات
iPhone 16 Pro,30,32000,39990,الجمعة للتكنولوجيا
Xiaomi Redmi Note 14,0,1500,1800,الإلكترونيات الحديثة
Sony WH-1000XM6,5,10500,14990,سوني مصر
Samsung Galaxy Tab S10,20,11500,14990,النور للتوريدات'''

result = chain.invoke({
    'sales_data': sample_sales,
    'inventory_data': sample_inventory,
    'language': 'Arabic'
})

print('— Executive Summary (AR) —')
print(result['summary'])
print('\n— Recommendations —')
for r in result['recommendations']:
    print(f"  {r['product']}: {r['action']} — {r['reason']} [{r['confidence']}%]")

In [ ]:
# Same analysis in English
result_en = chain.invoke({
    'sales_data': sample_sales,
    'inventory_data': sample_inventory,
    'language': 'English'
})
print('— English Version —')
print(result_en['summary'])

---
## Part 2: RAG — Vector Search Over Product Knowledge
Loads the product knowledge base into FAISS, retrieves context, answers questions grounded in it.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA

# ─── Product knowledge base (Egyptian market context) ───
product_kb = '''Product: Samsung Galaxy A56
Category: Smartphones | Price: 5,990 EGP | Supplier: النور للتوريدات
Key Features: 6.5-inch Super AMOLED, 50MP camera, 5000mAh battery, 8GB RAM
Market Position: Mid-range, strong competitor to Xiaomi Redmi Note 14
Target Audience: Egyptian youth, students, young professionals
Seasonality: High demand during back-to-school and Ramadan | Margin: ~15%

Product: iPhone 16 Pro
Category: Smartphones | Price: 39,990 EGP | Supplier: الجمعة للتكنولوجيا
Key Features: 6.3-inch LTPO OLED, 48MP camera, A18 Pro chip, 256GB
Market Position: Premium, brand-driven demand
Target Audience: High-income professionals, business owners
Seasonality: Peaks in Q4 after global launch | Margin: ~20%

Product: Xiaomi Redmi Note 14
Category: Smartphones | Price: 1,800 EGP | Supplier: الإلكترونيات الحديثة
Key Features: 6.67-inch AMOLED, 108MP camera, 5000mAh battery
Market Position: Budget king, highest volume seller
Target Audience: Students, budget-conscious buyers
Seasonality: Consistent year-round | Margin: ~10-12%

Product: Sony WH-1000XM6
Category: Headphones | Price: 14,990 EGP | Supplier: سوني مصر
Key Features: Industry-leading ANC, 40-hour battery, LDAC support
Market Position: Premium audio, niche
Target Audience: Audiophiles, frequent travelers
Seasonality: Higher during travel seasons | Margin: ~30%

Product: Samsung Galaxy Tab S10
Category: Tablets | Price: 14,990 EGP | Supplier: النور للتوريدات
Key Features: 12.4-inch Dynamic AMOLED, S Pen, DeX mode
Market Position: Premium tablet, alternative to iPad
Target Audience: Professionals, students, artists
Seasonality: Back-to-school, Ramadan | Margin: ~23%'''

docs = [Document(page_content=product_kb)]
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
chunks = splitter.split_documents(docs)
print(f'{len(chunks)} document chunks created')

In [ ]:
# ─── Embeddings + FAISS vector store ───
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(chunks, embeddings)
print('FAISS vector store created')

# ─── RAG chain ───
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={'k': 2})
)

query = 'Which product has the best margin for a small shop in Cairo to stock?'
answer = rag_chain.invoke({'query': query})
print(f'\nQ: {query}')
print(f'A: {answer["result"]}')

---
## Part 3: CrewAI — Multi-Agent Board Meeting
CEO, CFO, Marketing, and Inventory agents debate whether to stock a product.

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM

crew_llm = LLM(model='groq/llama-3.3-70b-versatile', temperature=0.3)

ceo = Agent(
    role='CEO',
    goal='Make the final strategic stocking decision',
    backstory='You run an electronics retail chain in Cairo. You weigh market '
              'opportunity, brand positioning, and long-term growth.',
    llm=crew_llm, verbose=True
)

cfo = Agent(
    role='CFO',
    goal='Evaluate the financial impact of stocking decisions',
    backstory='You are the finance chief. You care about margins, cash flow, '
              'and ROI. You hate inventory that ties up capital.',
    llm=crew_llm, verbose=True
)

marketing = Agent(
    role='Marketing Director',
    goal='Assess market demand and brand potential',
    backstory='You track Egyptian social media trends, competitor moves, and '
              'customer sentiment. You know what sells in Cairo.',
    llm=crew_llm, verbose=True
)

inventory_mgr = Agent(
    role='Inventory Manager',
    goal='Keep stock levels optimal and the warehouse efficient',
    backstory='You manage the Cairo warehouse. You know what is overstocked, '
              'which suppliers are reliable, and storage limits. You hate dead stock.',
    llm=crew_llm, verbose=True
)
print('4 agents ready')

In [ ]:
product_to_analyze = '''Product: Samsung Galaxy A56
Cost: 5,100 EGP | Selling Price: 5,990 EGP | Margin: ~15%
Monthly Sales: 45 units | Current Stock: 120 units
Supplier: النور للتوريدات (reliable, 3-day delivery)
Competition: Xiaomi Redmi Note 14 at 1,800 EGP, iPhone 16 Pro at 39,990 EGP
Market Trend: Strong mid-range demand in Egypt
Seasonality: High during back-to-school and Ramadan'''

t1 = Task(
    description=f'Analyze this product FINANCIALLY:\n{product_to_analyze}\n'
                'Focus on margin adequacy, capital tied up, ROI, overstocking risk.',
    expected_output='Financial assessment with numbers and a clear verdict',
    agent=cfo
)
t2 = Task(
    description=f'Analyze this product for MARKETING:\n{product_to_analyze}\n'
                'Focus on brand strength, demand trends, competitor positioning, sentiment.',
    expected_output='Marketing assessment with market insights and a verdict',
    agent=marketing
)
t3 = Task(
    description=f'Analyze this product for INVENTORY:\n{product_to_analyze}\n'
                'Focus on stock turnover, storage, supplier reliability, reorder timing.',
    expected_output='Inventory assessment with stock recommendations',
    agent=inventory_mgr
)
t4 = Task(
    description='Review the CFO, Marketing, and Inventory assessments. '
                'Make the FINAL DECISION: stock or not, and how many units. Give clear reasoning.',
    expected_output='Final decision with quantity and reasoning',
    agent=ceo
)

crew = Crew(
    agents=[cfo, marketing, inventory_mgr, ceo],
    tasks=[t1, t2, t3, t4],
    process=Process.sequential,
    verbose=True
)

print('=' * 60)
print('AI BOARD MEETING — ProductIQ')
print('=' * 60)
result = crew.kickoff()
print('\n' + '=' * 60)
print('FINAL DECISION')
print('=' * 60)
print(result)

---
## Part 4: What-If Simulator
Simple LangChain chain that simulates a pricing decision with honest assumptions.

In [ ]:
from langchain_core.prompts import PromptTemplate

whatif_prompt = PromptTemplate(
    input_variables=['product', 'price', 'change_type', 'change_value'],
    template='''You are a retail decision simulator for the Egyptian market (Cairo).

Product: {product}
Current Price: {price} EGP
Proposed Change: {change_type} by {change_value}

Estimate realistically:
1. demand_change_pct (number)
2. revenue_impact_egp (number, signed)
3. profit_impact_egp (number, signed)
4. risk_level (low / medium / high)
5. confidence_pct (0-100)
6. assumptions (short list of strings)

Return ONLY valid JSON with those keys.'''
)

sim_chain = whatif_prompt | llm | JsonOutputParser()

sim = sim_chain.invoke({
    'product': 'Samsung Galaxy A56',
    'price': '5,990',
    'change_type': 'price decrease',
    'change_value': '10%'
})

print('— What-If Simulation —')
print('Scenario: Samsung Galaxy A56 price -10%')
for k, v in sim.items():
    print(f'  {k}: {v}')

---
## Summary

| Operation | Tool | What It Does |
|-----------|------|--------------|
| **LangChain** | Groq `llama-3.3-70b-versatile` + JSON parser | Turns store data into structured recommendations (AR/EN) |
| **RAG** | FAISS + HuggingFace embeddings | Answers questions grounded in the product knowledge base |
| **CrewAI** | 4-agent board meeting | CEO / CFO / Marketing / Inventory debate a stocking decision |
| **What-If** | LangChain + JSON parser | Simulates price/discount scenarios with stated assumptions |